# Reproducing every figure and every reported number

This notebook runs inside the supplementary package and needs nothing outside it. No network, no
GPU, no model. Every cell reads the graded records that ship here and recomputes what the paper
reports from them.

It does three things, in order of how much they are worth.

1. **Provenance.** Checks that each judge's predictions are the ones the test seed was drawn
   against, by hash. This is the claim the whole design rests on and it is checkable here.
2. **Figures.** Regenerates every figure in the paper from the records.
3. **Agreement.** Recomputes the numbers the paper prints and compares them to what is printed,
   reporting any disagreement rather than asserting there is none.

Run it top to bottom. Anything that disagrees is reported as `MISMATCH` and the notebook does not
hide it.

In [ ]:
import gzip, hashlib, json, subprocess, sys
from pathlib import Path

# The package root, found from this notebook rather than from any absolute path.
HERE = Path.cwd()
PKG = next((p for p in [HERE, *HERE.parents] if (p / 'experiments').is_dir() and (p / 'MANIFEST.md').exists()), None)
assert PKG is not None, 'run this notebook from inside the supplementary package'
EXP = PKG / 'experiments'
BUILD = PKG / 'paper' / 'iclr2027' / 'build'
print('package root:', PKG)
print('gates present:', sorted(p.name for p in EXP.iterdir() if p.is_dir()))

PROBLEMS = []
def check(label, got, want, tol=5e-4):
    ok = (abs(got - want) <= tol) if isinstance(want, (int, float)) else (got == want)
    if not ok:
        PROBLEMS.append((label, got, want))
    print('%-52s records %-10s paper %-10s %s' % (label, got, want, 'ok' if ok else 'MISMATCH'))
    return ok

## 1. Provenance

The gate's claim is that the test stimuli did not exist when the predictions were written. What is
checkable after the fact is the next best thing: that the predictions graded are byte for byte the
ones pinned when the seed was drawn. `test_seed.json` records each judge's `predictions.json` by
sha256 at the moment of drawing, and the graded record carries the hash it actually used.

In [ ]:
for gate in ('G3e', 'G3f'):
    seed_rec = EXP / gate / 'run_record' / 'test_seed.json'
    verdicts = EXP / gate / 'run_record' / 'verdicts.json'
    if not (seed_rec.exists() and verdicts.exists()):
        print(gate, 'no sealed seed record in this package'); continue
    pinned = json.loads(seed_rec.read_text())['predictions_pinned']
    v = json.loads(verdicts.read_text())
    print('\n%s  test seed %s' % (gate, v.get('test_seed')))
    for judge, rec in v['judges'].items():
        want = pinned.get(judge, {}).get('sha256')
        got = rec.get('predictions_sha256')
        # And recompute the hash from the file that ships, so this is not two records agreeing
        # with each other while both differ from what is here.
        p = EXP / gate / 'run_record' / 'calibration' / (judge + '__served') / 'predictions.json'
        onfile = hashlib.sha256(p.read_bytes()).hexdigest() if p.exists() else None
        state = 'ok' if (want and got == want and onfile == want) else 'MISMATCH'
        if state != 'ok':
            PROBLEMS.append(('%s/%s provenance' % (gate, judge), got, want))
        print('  %-12s pinned %s  graded %s  on file %s  %s'
              % (judge, (want or '-')[:12], (got or '-')[:12], (onfile or '-')[:12], state))

In [ ]:
# --- G3g provenance: one seed for three cells, six predictions pinned by cell and judge -----------
gate = 'G3g'
seed_rec = EXP / gate / 'run_record' / 'test_seed.json'
verdicts = EXP / gate / 'run_record' / 'verdicts.json'
if not (seed_rec.exists() and verdicts.exists()):
    print(gate, 'no sealed seed record in this package')
else:
    pinned = json.loads(seed_rec.read_text())['predictions_pinned']
    v = json.loads(verdicts.read_text())
    print('\n%s  test seed %s  (one seed for every cell)' % (gate, v.get('test_seed')))
    for cell, judges in v['cells'].items():
        for judge, rec in judges.items():
            key = '%s/%s' % (cell, judge)
            want = pinned.get(key, {}).get('sha256')
            got = rec.get('predictions_sha256')
            p = EXP / gate / 'run_record' / cell / 'calibration' / (judge + '__served') / 'predictions.json'
            # The pin is the sha256 of the bytes on the machine that drew the seed (CRLF line
            # endings); this package ships the canonical LF form. The same content is accepted
            # under either convention, and the git blob hash, which is line-ending independent,
            # is checked against the pinned blob as well.
            raw = p.read_bytes() if p.exists() else b''
            lf = raw.replace(b'\r\n', b'\n')
            variants = {hashlib.sha256(b).hexdigest() for b in (raw, lf, lf.replace(b'\n', b'\r\n'))}
            blob = hashlib.sha1(b'blob %d\0' % len(lf) + lf).hexdigest()
            want_blob = pinned.get(key, {}).get('blob')
            onfile = want if want in variants else hashlib.sha256(raw).hexdigest()
            state = 'ok' if (want and got == want and want in variants and blob == want_blob) else 'MISMATCH'
            if state != 'ok':
                PROBLEMS.append(('%s/%s provenance' % (gate, key), got, want))
            print('  %-18s pinned %s  graded %s  on file %s  %s'
                  % (key, (want or '-')[:12], (got or '-')[:12], (onfile or '-')[:12], state))


In [ ]:
# --- G3h provenance: one prediction file, pinned by blob and by sha256 ----------------------------
gate = 'G3h'
seed_rec = EXP / gate / 'run_record' / 'test_seed.json'
grade = EXP / gate / 'run_record' / 'grade.json'
p = EXP / gate / 'run_record' / 'predictions.json'
if not (seed_rec.exists() and grade.exists() and p.exists()):
    print(gate, 'no sealed seed record in this package')
else:
    pin = json.loads(seed_rec.read_text())['predictions_pinned']
    got = json.loads(grade.read_text()).get('predictions_sha256')
    raw = p.read_bytes(); lf = raw.replace(b'\r\n', b'\n')
    variants = {hashlib.sha256(b).hexdigest() for b in (raw, lf, lf.replace(b'\n', b'\r\n'))}
    blob = hashlib.sha1(b'blob %d\0' % len(lf) + lf).hexdigest()
    ok = got in (pin['sha256'], pin.get('sha256_lf')) and (pin['sha256'] in variants or pin.get('sha256_lf') in variants) and blob == pin['blob']
    if not ok:
        PROBLEMS.append(('G3h provenance', got, pin['sha256']))
    print('\n%s  test seed %s' % (gate, json.loads(seed_rec.read_text())['drawn']))
    print('  pinned %s  graded %s  blob %s  %s' % (pin['sha256'][:12], (got or '-')[:12], blob[:12], 'ok' if ok else 'MISMATCH'))


In [ ]:
# --- G3j provenance, and G3i's calibration-only record ---------------------------------------------
for gate in ('G3j',):
    seed_rec = EXP / gate / 'run_record' / 'test_seed.json'
    grade = EXP / gate / 'run_record' / 'grade.json'
    p = EXP / gate / 'run_record' / 'predictions.json'
    if not (seed_rec.exists() and grade.exists() and p.exists()):
        print(gate, 'no sealed seed record in this package'); continue
    pin = json.loads(seed_rec.read_text())['predictions_pinned']
    got = json.loads(grade.read_text()).get('predictions_sha256')
    raw = p.read_bytes(); lf = raw.replace(b'\r\n', b'\n')
    variants = {hashlib.sha256(b).hexdigest() for b in (raw, lf, lf.replace(b'\n', b'\r\n'))}
    blob = hashlib.sha1(b'blob %d\0' % len(lf) + lf).hexdigest()
    ok = got in (pin['sha256'], pin.get('sha256_lf')) and (pin['sha256'] in variants or pin.get('sha256_lf') in variants) and blob == pin['blob']
    if not ok:
        PROBLEMS.append(('%s provenance' % gate, got, pin['sha256']))
    print('\n%s  test seed %s' % (gate, json.loads(seed_rec.read_text())['drawn']))
    print('  pinned %s  graded %s  blob %s  %s' % (pin['sha256'][:12], (got or '-')[:12], blob[:12], 'ok' if ok else 'MISMATCH'))
# G3i drew no seed: its calibration predicted no crossing inside its ladder, and the package ships
# that calibration and its predictions so the vacuity can be re-derived.
pi = EXP / 'G3i' / 'run_record' / 'predictions.json'
if pi.exists():
    v = json.loads(pi.read_text())
    check('G3i anti-vacuity from its calibration', v['anti_vacuity']['met'], False)
    check('G3i predicted reversal share at 192 tokens', round(v['reversal']['share'][-1], 3), 0.468, 1e-3)
    check('G3i test seed drawn', (EXP / 'G3i' / 'run_record' / 'test_seed.json').exists(), False)


## 2. Figures

Each script reads the graded records and writes into `paper/iclr2027/figures/`. They are the same
scripts that made the figures in the submitted PDF, unmodified.

In [ ]:
for script, what in (('make_v2_figures.py', 'Figures 1, 2 and 3'),
                     ('channel_fit.py', 'Figure 5 and Table 3, the numeric gate'),
                     ('make_figures.py', 'Figure 4, thresholds against budget')):
    p = BUILD / script
    if not p.exists():
        print('%-22s not in this package' % script); continue
    r = subprocess.run([sys.executable, str(p)], capture_output=True, text=True, cwd=str(BUILD))
    tail = (r.stdout or r.stderr).strip().splitlines()[-2:]
    print('%-22s %-38s rc=%d  %s' % (script, what, r.returncode, ' | '.join(tail)))
    if r.returncode != 0:
        PROBLEMS.append((script, 'rc=%d' % r.returncode, 'rc=0'))

figs = sorted((PKG / 'paper' / 'iclr2027' / 'figures').glob('*.pdf'))
print('\nfigures written:', [f.name for f in figs])

## 3. The numbers the paper prints

Recomputed from the graded records and compared with what the paper states. The paper's values are
written out here so a disagreement is visible rather than silent.

In [ ]:
# --- the worksheet gates, Section 4 -------------------------------------------------
g3c = EXP / 'G3c' / 'run_record'
paper_dev = {'qwen7b__full': 0.078, 'qwen7b__int4': 0.055, 'qwen14b__full': 0.072}
paper_tau = {'qwen7b__full': 0.876, 'qwen7b__int4': 0.908, 'qwen14b__full': 0.948}
for judge in ('qwen7b__full', 'qwen7b__int4', 'qwen14b__full'):
    f = g3c / ('grade_%s.json' % judge)
    if not f.exists():
        print(judge, 'no graded record'); continue
    v = json.loads(f.read_text())['verdicts']
    dev = max(r['max_abs_dev'] for r in v['J1_prediction'].values() if isinstance(r, dict))
    check('G3c %s largest deviation' % judge, round(dev, 3), paper_dev[judge], 1e-3)
    check('G3c %s Kendall tau' % judge, round(v['J3_budget_ordering']['kendall_tau'], 3), paper_tau[judge], 1e-3)

In [ ]:
# --- the served replication, and the judge whose ordering failed ---------------------
f = EXP / 'G3e' / 'run_record' / 'verdicts.json'
if f.exists():
    v = json.loads(f.read_text())
    worst = max(j['J1_max_dev'] for j in v['judges'].values())
    check('G3e largest deviation over its judges', round(worst, 3), 0.077, 1e-3)
    check('G3e gemma31b Kendall tau (the claim that failed)',
          round(v['judges']['gemma31b']['J3_kendall_tau'], 3), 0.804, 1e-3)
    check('G3e gemma31b read-outs at the ladder floor',
          v['judges']['gemma31b']['J3_predicted_at_ladder_floor'], 7)
    check('G3e gate verdict', v['gate'], 'FAIL')

In [ ]:
# --- the second stimulus family, Appendix F and Table 6 ------------------------------
f = EXP / 'G3f' / 'run_record' / 'verdicts.json'
if f.exists():
    v = json.loads(f.read_text())
    want = {'gemma31b': dict(dev=0.066, z=2.97, tau=0.869, floor=2),
            'gemma12b': dict(dev=0.124, z=3.55, tau=0.908, floor=0)}
    for judge, w in want.items():
        j = v['judges'][judge]
        check('G3f %s deviation' % judge, round(j['J1_max_dev'], 3), w['dev'], 1e-3)
        check('G3f %s noise units' % judge, round(j['J1_max_z'], 2), w['z'], 1e-2)
        check('G3f %s Kendall tau' % judge, round(j['J3_kendall_tau'], 3), w['tau'], 1e-3)
        check('G3f %s at the ladder floor' % judge, j['J3_predicted_at_ladder_floor'], w['floor'])
    check('G3f gate verdict', v['gate'], 'PASS')
    # The channel against the nominal rival, the range Table 6 quotes.
    eff = [r['effective'] for j in v['judges'].values() for r in j['J2_sse'].values()]
    nom = [r['nominal'] for j in v['judges'].values() for r in j['J2_sse'].values()]
    print('\nchannel squared error %.4f to %.4f, nominal %.4f to %.4f, channel wins %d of %d cells'
          % (min(eff), max(eff), min(nom), max(nom),
             sum(1 for a, b in zip(eff, nom) if a < b), len(eff)))

In [ ]:
# --- the third family, Appendix G and Table 7 ----------------------------------------------------
f = EXP / 'G3g' / 'run_record' / 'verdicts.json'
if f.exists():
    v = json.loads(f.read_text())
    want = {('trunc200', 'gemma31b'): dict(dev=0.083, z=2.81, tau=0.752, floor=0),
            ('trunc400', 'gemma31b'): dict(dev=0.059, z=2.72, tau=0.895, floor=4),
            ('full', 'gemma31b'): dict(dev=0.036, z=2.45, tau=0.641, floor=10),
            ('trunc200', 'gemma12b'): dict(dev=0.084, z=2.78, tau=0.791, floor=0),
            ('trunc400', 'gemma12b'): dict(dev=0.079, z=2.83, tau=0.758, floor=1),
            ('full', 'gemma12b'): dict(dev=0.035, z=3.08, tau=0.706, floor=8)}
    for (cell, judge), w in want.items():
        j = v['cells'][cell][judge]
        check('G3g %s %s deviation' % (cell, judge), round(j['J1_max_dev'], 3), w['dev'], 1e-3)
        check('G3g %s %s noise units' % (cell, judge), round(j['J1_max_z'], 2), w['z'], 1e-2)
        check('G3g %s %s Kendall tau' % (cell, judge), round(j['J3_kendall_tau'], 3), w['tau'], 1e-3)
        check('G3g %s %s at the ladder floor' % (cell, judge), j['J3_predicted_at_ladder_floor'], w['floor'])
    check('G3g gemma12b vacuous at 200 characters', v['cells']['trunc200']['gemma12b']['anti_vacuity_met'], False)
    check('G3g C1g', v['C1g'], 'PASS'); check('G3g C2g', v['C2g'], 'PASS')
    check('G3g C3g reported, not graded', v['C3g'], 'NOT GRADED'); check('G3g C5g', v['C5g'], 'PASS')
    check('G3g gate verdict', v['gate'], 'PASS')
    # The budget claim's numbers: greedy thresholds observed at 200 characters and at full length,
    # and predicted along the ladder, on gemma31b, as Appendix G quotes them.
    t = lambda cell, sc, k: v['cells'][cell]['gemma31b']['thresholds'][sc + '/argmax'][k]
    for sc, obs200, obsfull, pred in (('1-5', 1.82, 1.43, (1.84, 1.58, 1.35)),
                                      ('0-9', 1.82, 1.27, (1.68, 1.36, 1.13)),
                                      ('0-100', 1.65, 1.00, (1.60, 1.21, 1.00))):
        check('G3g greedy observed at 200, scale %s' % sc, round(t('trunc200', sc, 'observed'), 2), obs200, 1e-2)
        check('G3g greedy observed at full, scale %s' % sc, round(t('full', sc, 'observed'), 2), obsfull, 1e-2)
        for cell, p in zip(('trunc200', 'trunc400', 'full'), pred):
            check('G3g greedy predicted %s, scale %s' % (cell, sc), round(t(cell, sc, 'predicted'), 2), p, 1e-2)
    eff = [r['effective'] for c in v['cells'].values() for j in c.values() if j['anti_vacuity_met'] for r in j['J2_sse'].values()]
    nom = [r['nominal'] for c in v['cells'].values() for j in c.values() if j['anti_vacuity_met'] for r in j['J2_sse'].values()]
    print('\nchannel squared error %.4f to %.4f, nominal %.4f to %.4f, channel wins %d of %d graded cells'
          % (min(eff), max(eff), min(nom), max(nom), sum(1 for a, b in zip(eff, nom) if a < b), len(eff)))


In [ ]:
# --- the located crossing, Section 5 and Figure 3 ---------------------------------------------
f = EXP / 'G3h' / 'run_record' / 'grade.json'
if f.exists():
    g = json.loads(f.read_text())
    f2 = g['F2_crossover']; f3 = g['F3_rivals']['sse']; f1 = g['F1_additivity']
    check('G3h predicted crossing (tokens)', round(f2['predicted_budget']), 72)
    check('G3h observed crossing (tokens)', round(f2['observed_budget']), 57)
    check('G3h crossing deviation (noise units)', round(abs(f2['z']), 2), 0.62, 1e-2)
    check('G3h crossing spread (log2 budget)', round(f2['sd_difference'], 2), 0.52, 1e-2)
    check('G3h share at 48 tokens', round(f2['observed_shares'][3], 2), 0.35, 1e-2)
    check('G3h share at 64 tokens', round(f2['observed_shares'][4], 2), 0.60, 1e-2)
    zmax = max(abs(r['z']) for c in ('reversal', 'control') for r in f1[c])
    check('G3h largest additivity deviation', round(zmax, 2), 3.00, 1e-2)
    check('G3h additive squared error', round(f3['additive'], 1), 1.7, 1e-1)
    check('G3h evidence-only squared error', round(f3['evidence_only'], 1), 64.4, 1e-1)
    check('G3h cue-only squared error', round(f3['cue_only'], 1), 16.4, 1e-1)
    check('G3h gate verdict', g['gate'], 'PASS')


In [ ]:
# --- the second vendor's located crossing, Section 5 and Figure 3 ----------------------------
f = EXP / 'G3j' / 'run_record' / 'grade.json'
if f.exists():
    g = json.loads(f.read_text())
    f2 = g['F2_crossover']; f3 = g['F3_rivals']['sse']; f1 = g['F1_additivity']
    check('G3j predicted crossing (tokens)', round(f2['predicted_budget']), 187)
    check('G3j observed crossing (tokens)', round(f2['observed_budget']), 166)
    check('G3j crossing deviation (noise units)', round(abs(f2['z']), 2), 0.91, 1e-2)
    check('G3j crossing spread (log2 budget)', round(f2['sd_difference'], 2), 0.18, 1e-2)
    zmax = max(abs(r['z']) for c in ('reversal', 'control') for r in f1[c])
    check('G3j largest additivity deviation', round(zmax, 2), 2.83, 1e-2)
    check('G3j additive squared error', round(f3['additive'], 1), 3.9, 1e-1)
    check('G3j evidence-only squared error', round(f3['evidence_only'], 1), 78.4, 1e-1)
    check('G3j cue-only squared error', round(f3['cue_only'], 1), 95.5, 1e-1)
    check('G3j gate verdict', g['gate'], 'PASS')


In [ ]:
# --- Appendix D, where the predictive power enters -----------------------------------
f = EXP / 'G3c' / 'budget_ladder.json'
if f.exists():
    d = json.loads(f.read_text())
    check('ladder: share closed by the count of scores',
          round(100 * d['share_of_the_gap_closed_by_2_cardinality']), 22, 1)
    check('ladder: share closed by the information rate',
          round(100 * d['share_of_the_gap_closed_by_2b_bits']), 39, 1)
    check('ladder: share closed by the score positions',
          round(100 * d['share_of_the_gap_closed_by_3_codebook']), 22, 1)
    check('ladder: cells with room to explain', d['live_cells'][0], 17)
    check('ladder: cells graded', d['live_cells'][1], 18)
    check('ladder: rate beats the capacity in', d['cells_2b_bits_beats_nominal'][0], 12)
    check('ladder: channel is best in', d['cells_where_the_channel_is_best'][0], 18)

## 4. Re-grading from the raw records

The checks above compare the paper against the graded summaries. This one skips the summaries
and re-runs the grader itself over the score records that ship here, into a scratch copy, then
compares the verdicts it produces against the ones shipped. If they agree, nothing between the
raw scores and the reported verdicts is being taken on trust.

This is the check a reader should care about most, and it is the one the package could not
support until the score records for every gate were included.

In [ ]:
import shutil, tempfile, os

for gate, grader in (('G3f', 'G3e/g3e_grade.py'), ('G3e', 'G3e/g3e_grade.py')):
    gdir = EXP / gate
    rec = gdir / 'run_record'
    cfg = next(iter(gdir.glob('*config*.json')), None)
    shipped = rec / 'verdicts.json'
    script = EXP / grader
    if not (rec.is_dir() and cfg and shipped.exists() and script.exists()):
        print('%-5s not re-gradable from this package' % gate); continue
    # Grade into a scratch copy so the shipped record is never overwritten.
    tmp = Path(tempfile.mkdtemp()) / 'run_record'
    shutil.copytree(rec, tmp)
    (tmp / 'verdicts.json').unlink(missing_ok=True)
    r = subprocess.run([sys.executable, str(script), '--config', str(cfg), '--out', str(tmp)],
                       capture_output=True, text=True, cwd=str(gdir))
    if r.returncode != 0 or not (tmp / 'verdicts.json').exists():
        print('%-5s grader did not run: %s' % (gate, (r.stderr or r.stdout).strip()[-160:]))
        PROBLEMS.append(('%s re-grade' % gate, 'rc=%d' % r.returncode, 'rc=0'))
        continue
    got = json.loads((tmp / 'verdicts.json').read_text())
    want = json.loads(shipped.read_text())
    same_gate = got.get('gate') == want.get('gate')
    rows = []
    for judge, w in want['judges'].items():
        g = got['judges'].get(judge, {})
        agree = all(abs(float(g.get(k, -9)) - float(w[k])) < 1e-9
                    for k in ('J1_max_dev', 'J1_max_z', 'J3_kendall_tau')) and \
                all(g.get(k) == w[k] for k in ('J1', 'J2', 'J3', 'gate'))
        rows.append((judge, agree, g.get('gate'), w.get('gate')))
        if not agree:
            PROBLEMS.append(('%s/%s re-grade' % (gate, judge), g.get('gate'), w.get('gate')))
    ok = same_gate and all(a for _, a, _, _ in rows)
    print('%-5s re-graded from raw records: gate %s (shipped %s)  %s'
          % (gate, got.get('gate'), want.get('gate'), 'ok' if ok else 'MISMATCH'))
    for judge, agree, g, w in rows:
        print('        %-12s %s' % (judge, 'reproduced' if agree else 'DIFFERS (%s vs %s)' % (g, w)))
    shutil.rmtree(tmp.parent, ignore_errors=True)

In [ ]:
# --- G3g re-graded from the raw records with the registered orchestrator ---------------------------
gdir = EXP / 'G3g'; rec = gdir / 'run_record'; cfg = gdir / 'g3g_config.json'
shipped = rec / 'verdicts.json'; script = gdir / 'g3g_grade.py'
if not (rec.is_dir() and cfg.exists() and shipped.exists() and script.exists()):
    print('G3g   not re-gradable from this package')
else:
    tmp = Path(tempfile.mkdtemp()) / 'run_record'
    shutil.copytree(rec, tmp)
    (tmp / 'verdicts.json').unlink(missing_ok=True)
    r = subprocess.run([sys.executable, str(script), '--config', str(cfg), '--out', str(tmp)],
                       capture_output=True, text=True, cwd=str(gdir))
    if r.returncode != 0 or not (tmp / 'verdicts.json').exists():
        print('G3g   grader did not run: %s' % (r.stderr or r.stdout).strip()[-160:])
        PROBLEMS.append(('G3g re-grade', 'rc=%d' % r.returncode, 'rc=0'))
    else:
        got = json.loads((tmp / 'verdicts.json').read_text()); want = json.loads(shipped.read_text())
        ok = got.get('gate') == want.get('gate') and all(got.get(k) == want.get(k) for k in ('C1g', 'C2g', 'C3g', 'C5g'))
        for cell, judges in want['cells'].items():
            for judge, w in judges.items():
                g = got['cells'].get(cell, {}).get(judge, {})
                agree = all(abs(float(g.get(k, -9)) - float(w[k])) < 1e-9 for k in ('J1_max_dev', 'J1_max_z', 'J3_kendall_tau')) \
                    and all(g.get(k) == w[k] for k in ('J1', 'J2', 'cell_verdict'))
                ok &= agree
                if not agree:
                    PROBLEMS.append(('G3g/%s/%s re-grade' % (cell, judge), g.get('cell_verdict'), w.get('cell_verdict')))
                print('        %-18s %s' % ('%s/%s' % (cell, judge), 'reproduced' if agree else 'DIFFERS'))
        print('G3g   re-graded from raw records: gate %s (shipped %s)  %s' % (got.get('gate'), want.get('gate'), 'ok' if ok else 'MISMATCH'))
        shutil.rmtree(tmp.parent, ignore_errors=True)


In [ ]:
# --- G3h re-graded from the raw records with G3d's grader, unmodified ------------------------------
gdir = EXP / 'G3h'; rec = gdir / 'run_record'; cfg = gdir / 'flip_config_qwen7b.json'
script = EXP / 'G3d' / 'flip_grade.py'; shipped = rec / 'grade.json'
if not (rec.is_dir() and cfg.exists() and shipped.exists() and script.exists()):
    print('G3h   not re-gradable from this package')
else:
    tmp = Path(tempfile.mkdtemp())
    r = subprocess.run([sys.executable, str(script), 'grade', str(rec / 'test'), '--config', str(cfg),
                        '--predictions', str(rec / 'predictions.json'), '--out', str(tmp / 'grade.json')],
                       capture_output=True, text=True, cwd=str(gdir))
    if r.returncode != 0 or not (tmp / 'grade.json').exists():
        print('G3h   grader did not run: %s' % (r.stderr or r.stdout).strip()[-160:])
        PROBLEMS.append(('G3h re-grade', 'rc=%d' % r.returncode, 'rc=0'))
    else:
        got = json.loads((tmp / 'grade.json').read_text()); want = json.loads(shipped.read_text())
        ok = got.get('gate') == want.get('gate') and abs(got['F2_crossover']['z'] - want['F2_crossover']['z']) < 1e-9 \
            and abs(got['F3_rivals']['sse']['additive'] - want['F3_rivals']['sse']['additive']) < 1e-9
        print('G3h   re-graded from raw records: gate %s (shipped %s)  %s' % (got.get('gate'), want.get('gate'), 'ok' if ok else 'MISMATCH'))
        if not ok:
            PROBLEMS.append(('G3h re-grade', got.get('gate'), want.get('gate')))
        shutil.rmtree(tmp, ignore_errors=True)


In [ ]:
# --- G3j re-graded from the raw records with G3d's grader, unmodified ------------------------------
gdir = EXP / 'G3j'; rec = gdir / 'run_record'; cfg = gdir / 'flip_config_gemma12b.json'
script = EXP / 'G3d' / 'flip_grade.py'; shipped = rec / 'grade.json'
if not (rec.is_dir() and cfg.exists() and shipped.exists() and script.exists()):
    print('G3j   not re-gradable from this package')
else:
    tmp = Path(tempfile.mkdtemp())
    r = subprocess.run([sys.executable, str(script), 'grade', str(rec / 'test'), '--config', str(cfg),
                        '--predictions', str(rec / 'predictions.json'), '--out', str(tmp / 'grade.json')],
                       capture_output=True, text=True, cwd=str(gdir))
    if r.returncode != 0 or not (tmp / 'grade.json').exists():
        print('G3j   grader did not run: %s' % (r.stderr or r.stdout).strip()[-160:])
        PROBLEMS.append(('G3j re-grade', 'rc=%d' % r.returncode, 'rc=0'))
    else:
        got = json.loads((tmp / 'grade.json').read_text()); want = json.loads(shipped.read_text())
        ok = got.get('gate') == want.get('gate') and abs(got['F2_crossover']['z'] - want['F2_crossover']['z']) < 1e-9 \
            and abs(got['F3_rivals']['sse']['additive'] - want['F3_rivals']['sse']['additive']) < 1e-9
        print('G3j   re-graded from raw records: gate %s (shipped %s)  %s' % (got.get('gate'), want.get('gate'), 'ok' if ok else 'MISMATCH'))
        if not ok:
            PROBLEMS.append(('G3j re-grade', got.get('gate'), want.get('gate')))
        shutil.rmtree(tmp, ignore_errors=True)


In [ ]:
print('\n' + '=' * 78)
if PROBLEMS:
    print('%d DISAGREEMENT(S) between the records and what the paper prints:' % len(PROBLEMS))
    for label, got, want in PROBLEMS:
        print('  %-56s records %s  paper %s' % (label, got, want))
else:
    print('Every checked number in the paper was reproduced from the records in this package.')
print('=' * 78)

## What this notebook does not do

It does not re-run any judge. Doing that needs the models and, for three of them, a served API, and
the point of shipping the graded records is that a reader does not have to. What it establishes is
that the figures and the reported numbers follow from those records, and that the predictions
graded are the ones the seed was drawn against.

It also does not re-derive the bars. They were fixed by a null simulation before any gate ran, and
their record ships with G3c. A reader who wants to check that the bars were not moved should
compare each gate's `bars` block against G3c's, which every registration here says it takes by
reference.